ベース：白地図 CARTO Light No Labels

地点データ（Point）：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr

地域線：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

分析手法：GeoPandas Spatial Join

対象地域：
・Syria (National Scale)

その他：
・PointデータとGovernorateポリゴンを重ね合わせる
・GeoPandas spatial join により、各PointがどのGovernorate内に位置するかを判定
・地点データへGovernorate名を付与し、空間的な所属関係を確認

In [1]:
# Core Libraries
# ベクタデータ処理に必要なライブラリを読み込む

import geopandas as gpd

import folium

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ETL Extract
# PointデータとGovernorate Polygonを読み込む

points = gpd.read_file("syr_adminpoints.geojson")
admin1 = gpd.read_file("syr_admin1.geojson")

In [3]:
# Check
# spatial join前にCRSが一致しているか確認する

print(points.crs)
print(admin1.crs)

EPSG:4326
EPSG:4326


In [4]:
# Check
# Pointデータの属性列とGeometry構造を確認する

print(points.head())

   admin_level                  name                      name1 name2 name3  \
0            0  Syrian Arab Republic  الجمهورية العربية السورية  None  None   
1            1            Al-Hasakeh                     الحسكة  None  None   
2            1                Aleppo                        حلب  None  None   
3            1              Ar-Raqqa                      الرقة  None  None   
4            1             As-Sweida                   السويداء  None  None   

     x_coord    y_coord adm4_name adm4_name1 adm4_name2  ... adm0_name3  \
0  38.576577  34.815880      None       None       None  ...       None   
1  40.557556  36.427519      None       None       None  ...       None   
2  37.395163  36.146897      None       None       None  ...       None   
3  38.920586  36.023036      None       None       None  ...       None   
4  36.894841  32.755555      None       None       None  ...       None   

  adm0_pcode   valid_on valid_to version lang lang1 lang2 lang3  \
0      

In [5]:
# ETL Transform
# spatial joinを実行する

joined = gpd.sjoin(
    points,
    admin1,
    predicate="within"
)

In [6]:
# Check
# spatial join結果を確認する
# PointへGovernorate属性が付与されたか確認する

print(joined.head())

   admin_level                  name                      name1 name2 name3  \
0            0  Syrian Arab Republic  الجمهورية العربية السورية  None  None   
1            1            Al-Hasakeh                     الحسكة  None  None   
2            1                Aleppo                        حلب  None  None   
3            1              Ar-Raqqa                      الرقة  None  None   
4            1             As-Sweida                   السويداء  None  None   

     x_coord    y_coord adm4_name adm4_name1 adm4_name2  ... valid_to_right  \
0  38.576577  34.815880      None       None       None  ...           None   
1  40.557556  36.427519      None       None       None  ...           None   
2  37.395163  36.146897      None       None       None  ...           None   
3  38.920586  36.023036      None       None       None  ...           None   
4  36.894841  32.755555      None       None       None  ...           None   

      area_sqkm version_right lang_right lang1_rig

In [7]:
# Check
# 「地点名 → 所属Governorate」の対応表を確認する

print(joined[["name", "adm1_ref_name"]])

                     name   adm1_ref_name
0    Syrian Arab Republic            Homs
1              Al-Hasakeh      Al-Hasakeh
2                  Aleppo          Aleppo
3                Ar-Raqqa        Ar-Raqqa
4               As-Sweida       As-Sweida
..                    ...             ...
344         Wadi El-oyoun            Hama
345            Ya'robiyah      Al-Hasakeh
346               Yabroud  Rural Damascus
347                Zarbah          Aleppo
348                Ziyara            Hama

[349 rows x 2 columns]


349地点に対して、所属Governorate（adm1_ref_name）の振り分けが完了している

In [8]:
# Map Setup
# 白地図ベースを作成

m = folium.Map(
    location=[34.8, 38.5], 
    zoom_start=7,
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
)

In [9]:
# ETL Transform
# Folium表示用に必要な列だけ残す

admin1_map = admin1[["adm1_name", "geometry"]]

In [10]:
# ETL Load
# 行政区ポリゴンを地図へ追加

folium.GeoJson(
    admin1_map,
    name="Governorates",
    style_function=lambda x: {
        "color": "gray",
        "weight": 1,
        "fillOpacity": 0
    }
).add_to(m)

In [11]:
# ETL Load
# Governorate属性を付与したPointデータを地図へ追加

for _, row in joined.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color="#3c4d70",
        fill=True,
        fillColor="#3c4d70",
        fillOpacity=0.7,
        popup=f"{row['name']} / {row['adm1_ref_name']}"
    ).add_to(m)

In [12]:
# ETL Load
# HTMLマップとして保存

m.save("13_syria_vector_spatialjoin.html")